# Gate 0B - Raw Acquisition and Quality Audit

This notebook owns the Gate 0B data lifecycle for Phase 1:

1. **Acquire** the smallest defensible first raw files (an OpenStreetMap walk
   graph inside the Gate 0 boundary, and the OS Open Greenspace `SP` extract).
2. **Audit** those persisted raw files against the forensic Quality Audit
   Checklist (CRS, geometry validity, connectivity, boundary coverage,
   anchor snapping, cross-source checks).
3. **Explore** the realities of the study area with maps and charts.

It does **not** score routes, rank interventions, or make funding-facing claims.
Acquisition is idempotent: existing raw files are reused, never re-downloaded.

## Acquisition Rules

- Every raw file must be recorded in `raw_data_manifest_phase1.csv`.
- Every raw file must have a SHA-256 checksum.
- Source-published MD5 checks must be verified where available.
- Source acquisition status can advance to `raw_acquired_pending_quality_audit`, but quality, cross-check, and funding gates remain closed.
- These files are raw evidence inputs, not conclusions.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import geopandas as gpd
import osmnx as ox
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PHASE1_ROOT = PROJECT_ROOT.parent
else:
    PHASE1_ROOT = PROJECT_ROOT / "phase1_spinelens_ai"

sys.path.insert(0, str(PHASE1_ROOT / "src"))

from spinelens.gate0b import (  # noqa: E402
    build_manifest_row,
    download_file_with_checks,
    mark_sources_raw_acquired,
    md5_file,
    raw_data_manifest_fieldnames,
    read_csv_rows,
    sha256_file,
    upsert_manifest_rows,
    utc_now_iso,
    write_csv_rows,
)

DATA = PHASE1_ROOT / "data"
RAW = DATA / "raw"
INTERIM = DATA / "interim"
REPORTS = PHASE1_ROOT / "outputs" / "reports"
RAW.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

BOUNDARY_PATH = INTERIM / "study_area_boundary_phase1.geojson"
OS_CANDIDATES_PATH = INTERIM / "gate0b_os_download_candidates.csv"
MANIFEST_PATH = DATA / "raw_data_manifest_phase1.csv"
ACQUISITION_PATH = DATA / "source_acquisition_status_phase1.csv"
ACQUISITION_NOTE_PATH = REPORTS / "gate0b_controlled_raw_acquisition_note.md"

manifest_rows = read_csv_rows(MANIFEST_PATH)
acquisition_rows = read_csv_rows(ACQUISITION_PATH)
pd.DataFrame(acquisition_rows).query("source_id in ['osm_network', 'os_open_greenspace']")

## Load Gate 0 Boundary

The OSM graph is extracted only inside the provisional Gate 0 boundary. This keeps the first acquisition small and relevant.

In [ ]:
boundary_gdf = gpd.read_file(BOUNDARY_PATH).to_crs(4326)
boundary_polygon = boundary_gdf.geometry.iloc[0]

boundary_summary = {
    "boundary_path": str(BOUNDARY_PATH.relative_to(PHASE1_ROOT)),
    "crs": str(boundary_gdf.crs),
    "geometry_type": boundary_polygon.geom_type,
    "bounds": tuple(round(value, 6) for value in boundary_polygon.bounds),
}
boundary_summary

## Acquire OSM Walk Graph

This is the first raw pedestrian-network extract. It is volunteered data and must be validated before it can support design or funding claims.

In [ ]:
osm_raw_dir = RAW / "osm_network"
osm_raw_dir.mkdir(parents=True, exist_ok=True)
osm_graph_path = osm_raw_dir / "gate0b_osm_walk_graph.graphml"

ox.settings.use_cache = True
ox.settings.cache_folder = str(osm_raw_dir / "osmnx_cache")
ox.settings.requests_timeout = 180
ox.settings.overpass_rate_limit = True

if osm_graph_path.exists():
    # Idempotent: reuse the immutable raw extract instead of re-downloading.
    osm_graph = ox.load_graphml(osm_graph_path)
else:
    osm_graph = ox.graph_from_polygon(
        boundary_polygon,
        network_type="walk",
        simplify=True,
        retain_all=True,
        truncate_by_edge=True,
    )
    ox.save_graphml(osm_graph, filepath=osm_graph_path)

osm_nodes, osm_edges = ox.graph_to_gdfs(osm_graph)
osm_summary = {
    "raw_file": str(osm_graph_path.relative_to(PHASE1_ROOT)).replace("\\", "/"),
    "nodes": int(len(osm_nodes)),
    "edges": int(len(osm_edges)),
    "file_size_bytes": osm_graph_path.stat().st_size,
    "sha256": sha256_file(osm_graph_path),
}
osm_summary

## Acquire OS Open Greenspace SP

This is a small authoritative context layer for checking the Ryder Street grassland/pavilion hypothesis. It is not a land-ownership or buildability conclusion.

In [ ]:
os_candidates = pd.read_csv(OS_CANDIDATES_PATH)
greenspace_candidates = os_candidates[
    (os_candidates["source_id"] == "os_open_greenspace")
    & (os_candidates["area"] == "SP")
    & (os_candidates["format"].str.contains("Shapefile", na=False))
    & (os_candidates["download_decision"] == "safe_small_candidate_for_pavilion_context")
]

if len(greenspace_candidates) != 1:
    raise ValueError(f"Expected one OS Open Greenspace SP shapefile candidate, found {len(greenspace_candidates)}")

greenspace_candidate = greenspace_candidates.iloc[0].to_dict()
greenspace_raw_dir = RAW / "os_open_greenspace"
greenspace_path = greenspace_raw_dir / greenspace_candidate["file_name"]

if greenspace_path.exists():
    # Idempotent: keep the immutable raw file; just recompute its summary.
    greenspace_local_md5 = md5_file(greenspace_path)
else:
    greenspace_download = download_file_with_checks(
        url=greenspace_candidate["download_url"],
        output_path=greenspace_path,
        expected_md5=greenspace_candidate["md5"],
        max_bytes=10_000_000,
        timeout_seconds=90,
    )
    greenspace_local_md5 = greenspace_download.md5

greenspace_summary = {
    "raw_file": str(greenspace_path.relative_to(PHASE1_ROOT)).replace("\\", "/"),
    "file_size_bytes": greenspace_path.stat().st_size,
    "sha256": sha256_file(greenspace_path),
    "source_md5": greenspace_candidate["md5"],
    "local_md5": greenspace_local_md5,
}
greenspace_summary

## Update Manifest And Acquisition Ledger

The manifest records raw files. The acquisition ledger advances only to raw-acquired/pending-quality-audit.

In [ ]:
new_manifest_rows = [
    build_manifest_row(
        manifest_id="gate0b_osm_walk_graph_boundary_v0",
        source_id="osm_network",
        raw_file_path=osm_graph_path,
        phase1_root=PHASE1_ROOT,
        download_url="https://overpass-api.de/api/interpreter",
        source_publication_date="continuous",
        source_version="OSMnx walk-network extract inside Gate 0 boundary",
        license_name="ODbL",
        provenance_note=(
            f"Extracted with OSMnx from Gate 0 boundary; "
            f"nodes={osm_summary['nodes']}; edges={osm_summary['edges']}; "
            "volunteered data requiring validation."
        ),
    ),
    build_manifest_row(
        manifest_id="gate0b_os_open_greenspace_sp_shapefile",
        source_id="os_open_greenspace",
        raw_file_path=greenspace_path,
        phase1_root=PHASE1_ROOT,
        download_url=greenspace_candidate["download_url"],
        source_publication_date="October 2025 stated in source registry",
        source_version="OpenGreenspace SP ESRI Shapefile",
        license_name="OGL",
        provenance_note=(
            f"Downloaded from OS Downloads API metadata candidate; "
            f"source_md5={greenspace_candidate['md5']}; local_md5={greenspace_summary['local_md5']}."
        ),
    ),
]

updated_manifest_rows = upsert_manifest_rows(manifest_rows, new_manifest_rows)
write_csv_rows(MANIFEST_PATH, updated_manifest_rows, raw_data_manifest_fieldnames())

updated_acquisition_rows = mark_sources_raw_acquired(
    acquisition_rows,
    {"osm_network", "os_open_greenspace"},
)
write_csv_rows(ACQUISITION_PATH, updated_acquisition_rows, list(acquisition_rows[0].keys()))

pd.DataFrame(updated_manifest_rows)

## Acquisition Note

This note is the review checkpoint before any quality audit, graph scoring, tactical-corridor scoring, or digital wayfinder content generation.

In [ ]:
checked_at = utc_now_iso()
note = "\n".join([
    "# Gate 0B Controlled Raw Acquisition Note",
    "",
    f"Generated: {checked_at}",
    "",
    "## Decision",
    "",
    "The first controlled raw acquisition is complete. The project now has an OSM walk-network extract and a small OS Open Greenspace SP raw file. These are inputs for quality audit, not route recommendations.",
    "",
    "## Acquired Files",
    "",
    "| Source | File | Bytes | SHA-256 |",
    "|---|---|---:|---|",
    f"| OSM walk network | {osm_summary['raw_file']} | {osm_summary['file_size_bytes']} | {osm_summary['sha256']} |",
    f"| OS Open Greenspace SP | {greenspace_summary['raw_file']} | {greenspace_summary['file_size_bytes']} | {greenspace_summary['sha256']} |",
    "",
    "## OSM Graph Summary",
    "",
    f"- Nodes: {osm_summary['nodes']}",
    f"- Edges: {osm_summary['edges']}",
    "- Scope: Gate 0 provisional boundary only.",
    "- Caution: OSM is volunteered data and must be cross-checked before use in funding claims.",
    "",
    "## Gate Status",
    "",
    "Raw acquisition has advanced for `osm_network` and `os_open_greenspace`, but both remain pending quality audit. No source is funding-ready.",
    "",
    "## Next Controlled Step",
    "",
    "Run a quality audit on the OSM graph and OS Greenspace file: inspect CRS, geometry validity, graph connectivity, route-origin snapping, candidate pavilion context, and obvious missing pedestrian links. Only after that should route-family experiments begin.",
])

ACQUISITION_NOTE_PATH.write_text(note, encoding="utf-8")
print(note)

## Quality Audit and Forensic Exploration (Evidence Level 2 -> 3)

The acquired OSM walk graph and OS Open Greenspace extract are at **Evidence Level 2**
("raw acquired", internal research only). This section runs the forensic Quality
Audit Checklist against the **persisted raw files**, cross-checks them, and
visualises the realities of the Phase 1 study area.

A source advances to **Level 3** ("quality audited", descriptive claims only) only
if its data-quality checks pass. No funding-facing claim is unlocked here.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import zipfile
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import Point

from spinelens.spatial import audit

# Forensic principle: audit the persisted artifact, not the in-memory graph.
audit_graph = ox.load_graphml(osm_graph_path)
audit_nodes, audit_edges = ox.graph_to_gdfs(audit_graph)
node_coords = {nid: (float(d["y"]), float(d["x"])) for nid, d in audit_graph.nodes(data=True)}

# Greenspace site polygons from the immutable raw zip.
gs_names = zipfile.ZipFile(greenspace_path).namelist()
gs_site_name = [n for n in gs_names if n.lower().replace(" ", "").endswith("greenspacesite.shp")][0]
greenspace = gpd.read_file(f"zip://{greenspace_path.as_posix()}!{gs_site_name}")

# Gate 0 / Gate 0A ledgers.
anchors = read_csv_rows(DATA / "study_area_anchors_phase1.csv")
route_families = read_csv_rows(DATA / "route_families_phase1.csv")

# Local-only figure outputs (outputs/ is gitignored).
FIG_DIR = PHASE1_ROOT / "outputs" / "reports" / "gate0b_audit_media"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR = PHASE1_ROOT / "outputs" / "maps"
MAPS_DIR.mkdir(parents=True, exist_ok=True)

ORIGIN_IDS = {"colmore_row", "new_street_station", "moor_street_queensway",
              "snow_hill_station", "dartmouth_middleway_nechells"}

def role_color(aid):
    if aid in ORIGIN_IDS:
        return "#1a73e8"            # inbound origins (blue)
    if aid == "ryder_street_pavilion_search_area":
        return "#f59e0b"            # gateway (amber)
    if aid == "dartmouth_jennens_crossing":
        return "#d93025"            # priority barrier (red)
    return "#34a853"               # onward anchors (green)

print(f"graph: nodes={audit_graph.number_of_nodes()} edges={audit_graph.number_of_edges()}")
print(f"greenspace: sites={len(greenspace)} crs={greenspace.crs}")
print(f"anchors={len(anchors)} route_families={len(route_families)}")

### 1. CRS, geometry validity, integrity

In [ ]:
osm_crs = str(audit_nodes.crs)
gs_crs = str(greenspace.crs)

edge_invalid = int((~audit_edges.geometry.is_valid).sum())
edge_empty = int(audit_edges.geometry.is_empty.sum())
gs_invalid = int((~greenspace.geometry.is_valid).sum())
gs_empty = int(greenspace.geometry.is_empty.sum())
edge_len = audit.summary_stats(audit_edges["length"].tolist())

integrity = pd.DataFrame([
    {"check": "OSM CRS known", "value": osm_crs},
    {"check": "Greenspace CRS known", "value": gs_crs},
    {"check": "OSM invalid / empty edges", "value": f"{edge_invalid} / {edge_empty}"},
    {"check": "Greenspace invalid / empty geoms", "value": f"{gs_invalid} / {gs_empty}"},
    {"check": "Edge length m (min/median/max)", "value": f"{edge_len['min']} / {edge_len['median']} / {edge_len['max']}"},
    {"check": "Total network length (km)", "value": round(edge_len["total"] / 1000, 2)},
])
display(integrity)

### 2. Graph connectivity

In [ ]:
undirected = audit_graph.to_undirected()
components = sorted(nx.connected_components(undirected), key=len, reverse=True)
largest = components[0]
largest_frac = len(largest) / audit_graph.number_of_nodes()
degrees = [d for _, d in undirected.degree()]
deg = audit.degree_stats(degrees)

connectivity = {
    "components": len(components),
    "largest_component_nodes": len(largest),
    "largest_component_fraction": round(largest_frac, 4),
    "nodes_outside_largest": audit_graph.number_of_nodes() - len(largest),
    "dead_ends": deg["dead_ends"],
    "decision_nodes_deg_ge_4": deg["decision_nodes"],
}
connectivity

### 3. Boundary coverage

Nodes can fall just outside the boundary because the extract uses `truncate_by_edge`.

In [ ]:
nodes_in = int(audit_nodes.within(boundary_polygon).sum())
coverage_frac = nodes_in / len(audit_nodes)
coverage = {
    "nodes_in_boundary": nodes_in,
    "nodes_total": len(audit_nodes),
    "coverage_fraction": round(coverage_frac, 4),
    "graph_bounds": [round(float(v), 5) for v in audit_nodes.total_bounds],
    "boundary_bounds": [round(float(v), 5) for v in boundary_gdf.total_bounds],
}
coverage

### 4. Anchor snapping to the network

In [ ]:
snap_df = pd.DataFrame(audit.snap_report(node_coords, anchors))
display(snap_df)

poor = snap_df[snap_df["snap_quality"] == "poor"]["anchor_id"].tolist()
key_origins = snap_df[snap_df["anchor_id"].isin(ORIGIN_IDS | {"ryder_street_pavilion_search_area"})]
key_origins_ok = bool((key_origins["snap_quality"] != "poor").all())
{"poor_snaps": poor, "key_origins_and_gateway_snap_ok": key_origins_ok}

### 5. Cross-source checks: route ledger <-> anchors, greenspace <-> pavilion

In [ ]:
anchor_ids = [a["anchor_id"] for a in anchors]
ref_df = pd.DataFrame(audit.resolve_route_family_references(route_families, anchor_ids))
display(ref_df)
unresolved_families = ref_df[~ref_df["resolves"]]["route_family_id"].tolist()

# Greenspace context around the Ryder Street pavilion search area (metric CRS).
pav = next(a for a in anchors if a["anchor_id"] == "ryder_street_pavilion_search_area")
pav_pt = gpd.GeoSeries([Point(float(pav["longitude"]), float(pav["latitude"]))], crs=4326).to_crs(27700).iloc[0]
gs_near = greenspace.copy()
gs_near["dist_m"] = gs_near.geometry.distance(pav_pt).round(1)
gs_near = gs_near[gs_near["dist_m"] <= 250].sort_values("dist_m")
display(gs_near[["function", "distName1", "dist_m"]].head(10))

# Characterise greenspace even when none sits at the pavilion itself.
nearest_gs_m = round(float(greenspace.geometry.distance(pav_pt).min()), 1)
boundary_27700 = boundary_gdf.to_crs(27700).geometry.iloc[0]
gs_in_boundary = int(greenspace.intersects(boundary_27700).sum())

{"unresolved_route_families": unresolved_families,
 "greenspace_sites_within_250m_of_pavilion": int(len(gs_near)),
 "nearest_greenspace_to_pavilion_m": nearest_gs_m,
 "greenspace_sites_in_study_boundary": gs_in_boundary}

## Forensic visuals

The realities of the city-core to B-KQ approach: network, greenspace, anchors, connectivity, and the gateway/barrier context.

In [ ]:
# Shared 4326 views
edges_4326 = audit_edges.to_crs(4326)
nodes_4326 = audit_nodes.to_crs(4326)
gs_4326 = greenspace.to_crs(4326)
bnd_4326 = boundary_gdf.to_crs(4326)
minx, miny, maxx, maxy = bnd_4326.total_bounds
gs_view = gs_4326.cx[minx:maxx, miny:maxy]
ASPECT = 1 / np.cos(np.radians((miny + maxy) / 2))

# FIGURE 1 - study area overview
fig, ax = plt.subplots(figsize=(11, 9))
gs_view.plot(ax=ax, color="#7bb274", alpha=0.45, edgecolor="none", zorder=1)
edges_4326.plot(ax=ax, color="#9aa0a6", linewidth=0.5, zorder=2)
bnd_4326.boundary.plot(ax=ax, color="#1f1f1f", linewidth=1.5, linestyle="--", zorder=3)
for a in anchors:
    lon, lat = float(a["longitude"]), float(a["latitude"])
    ax.scatter(lon, lat, s=80, color=role_color(a["anchor_id"]), edgecolor="white", linewidth=0.8, zorder=5)
    ax.annotate(a["anchor_name"], (lon, lat), fontsize=7, xytext=(4, 4),
                textcoords="offset points", zorder=6)
ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy); ax.set_aspect(ASPECT)
ax.set_title("Phase 1 study area: OSM walk network, greenspace, provisional anchors")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
fig.savefig(FIG_DIR / "fig1_study_area_overview.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURE 2 - graph connectivity
in_largest = nodes_4326.index.isin(list(largest))
fig, ax = plt.subplots(figsize=(11, 9))
edges_4326.plot(ax=ax, color="#d6d6d6", linewidth=0.4, zorder=1)
nodes_4326[in_largest].plot(ax=ax, color="#1a73e8", markersize=2, zorder=2,
                            label=f"largest component ({len(largest)} nodes)")
if (~in_largest).any():
    nodes_4326[~in_largest].plot(ax=ax, color="#d93025", markersize=14, zorder=3,
                                 label=f"fragments ({connectivity['nodes_outside_largest']} nodes)")
ax.legend(loc="upper right"); ax.set_aspect(ASPECT)
ax.set_title("Graph connectivity: largest connected component vs fragments")
fig.savefig(FIG_DIR / "fig2_connectivity.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURE 3 - distributions (snap, edge length, node degree)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
sd = snap_df.sort_values("snap_distance_m")
axes[0].barh(sd["anchor_id"], sd["snap_distance_m"],
             color=[role_color(a) for a in sd["anchor_id"]])
axes[0].axvline(audit.SNAP_GOOD_M, color="green", ls="--", lw=1, label="good <=25m")
axes[0].axvline(audit.SNAP_ACCEPTABLE_M, color="orange", ls="--", lw=1, label="acceptable <=75m")
axes[0].set_title("Anchor snap distance to network (m)")
axes[0].legend(fontsize=7)
axes[1].hist(audit_edges["length"], bins=40, color="#1a73e8")
axes[1].set_title("Edge length distribution (m)")
axes[1].set_xlabel("length (m)"); axes[1].set_ylabel("edges")
axes[2].hist(degrees, bins=range(1, 9), color="#7bb274", align="left", rwidth=0.85)
axes[2].set_title("Node degree distribution")
axes[2].set_xlabel("degree"); axes[2].set_ylabel("nodes")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_distributions.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURE 4 - gateway and barrier context
focus_ids = {"ryder_street_pavilion_search_area", "dartmouth_jennens_crossing",
             "moor_street_queensway", "millennium_point", "aston_university",
             "dartmouth_middleway_nechells"}
fl = [a for a in anchors if a["anchor_id"] in focus_ids]
lons = [float(a["longitude"]) for a in fl]; lats = [float(a["latitude"]) for a in fl]
pad = 0.0025
fminx, fmaxx = min(lons) - pad, max(lons) + pad
fminy, fmaxy = min(lats) - pad, max(lats) + pad
fig, ax = plt.subplots(figsize=(11, 9))
gs_4326.cx[fminx:fmaxx, fminy:fmaxy].plot(ax=ax, color="#7bb274", alpha=0.5, zorder=1)
edges_4326.cx[fminx:fmaxx, fminy:fmaxy].plot(ax=ax, color="#9aa0a6", linewidth=0.8, zorder=2)
for a in fl:
    lon, lat = float(a["longitude"]), float(a["latitude"])
    ax.scatter(lon, lat, s=110, color=role_color(a["anchor_id"]), edgecolor="white", linewidth=1, zorder=5)
    ax.annotate(a["anchor_name"], (lon, lat), fontsize=8, xytext=(5, 5),
                textcoords="offset points", zorder=6)
ax.set_xlim(fminx, fmaxx); ax.set_ylim(fminy, fmaxy); ax.set_aspect(ASPECT)
ax.set_title("Gateway and barrier context: Ryder Street pavilion and Dartmouth / Jennens crossing")
fig.savefig(FIG_DIR / "fig4_gateway_barrier_context.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Interactive map (saved locally; outputs/ is gitignored)
import folium

m = folium.Map(location=[52.484, -1.892], zoom_start=15, tiles="cartodbpositron")
folium.GeoJson(bnd_4326.to_json(), name="study boundary",
               style_function=lambda f: {"color": "#1f1f1f", "weight": 2, "dashArray": "5,5", "fill": False}).add_to(m)
folium.GeoJson(gs_view.to_json(), name="greenspace",
               style_function=lambda f: {"color": "#7bb274", "fillColor": "#7bb274", "weight": 0, "fillOpacity": 0.4}).add_to(m)
folium.GeoJson(edges_4326[["geometry"]].to_json(), name="walk network",
               style_function=lambda f: {"color": "#9aa0a6", "weight": 1}).add_to(m)
for a in anchors:
    folium.CircleMarker([float(a["latitude"]), float(a["longitude"])], radius=6,
                        color=role_color(a["anchor_id"]), fill=True, fill_opacity=0.9,
                        popup=f"{a['anchor_name']} ({a['anchor_id']})").add_to(m)
folium.LayerControl().add_to(m)
map_path = MAPS_DIR / "gate0b_quality_audit.html"
m.save(str(map_path))
print("saved interactive map:", map_path.relative_to(PHASE1_ROOT))
m

## Verdict and ledger update

In [ ]:
edge_invalid_frac = edge_invalid / max(len(audit_edges), 1)
gs_invalid_frac = gs_invalid / max(len(greenspace), 1)

# Routing feasibility: do the key origins + gateway land in one connected component?
key_ids = ORIGIN_IDS | {"ryder_street_pavilion_search_area"}
key_nodes = snap_df[snap_df["anchor_id"].isin(key_ids)]["nearest_node_id"].tolist()
key_anchors_connected = all(n in largest for n in key_nodes)
disconnected_key_anchors = snap_df[
    snap_df["anchor_id"].isin(key_ids) & ~snap_df["nearest_node_id"].isin(largest)
]["anchor_id"].tolist()

# Data-quality checks (drive Level 2 -> 3). Site/context findings are reported
# separately below and never block a source's data-quality verdict.
osm_checks = {
    "crs_known": "epsg" in osm_crs.lower(),
    "geometries_valid": edge_invalid_frac <= 0.005 and edge_empty == 0,
    "covers_study_boundary": coverage_frac >= 0.85,
    "key_anchors_mutually_connected": key_anchors_connected,
}
gs_checks = {
    "crs_known": "epsg" in gs_crs.lower(),
    "geometries_valid": gs_invalid_frac <= 0.005 and gs_empty == 0,
    "covers_study_area": gs_in_boundary >= 1,
}
osm_verdict = audit.evidence_verdict(osm_checks)
gs_verdict = audit.evidence_verdict(gs_checks)
print("osm_network        ->", osm_verdict)
print("os_open_greenspace ->", gs_verdict)
print("key_anchors_mutually_connected:", key_anchors_connected,
      "| disconnected:", disconnected_key_anchors)
print("nearest greenspace to pavilion (m):", nearest_gs_m,
      "| greenspace sites in study boundary:", gs_in_boundary)

In [ ]:
audit_ts = utc_now_iso()

def apply_audit(rows, source_id, verdict):
    out = []
    for r in rows:
        r = dict(r)
        if r["source_id"] == source_id:
            r["quality_audited"] = "yes" if verdict["passed"] else "no"
            r["forensic_status"] = verdict["forensic_status"]
            r["evidence_level"] = str(verdict["evidence_level"])
            r["can_support_funding_claim"] = "no"
            r["next_action"] = (
                "Cross-check against a second authoritative source, then refine provisional anchors."
                if verdict["passed"]
                else f"Resolve failed checks: {', '.join(verdict['failed_checks'])}."
            )
            tag = f"Gate 0B quality audit {audit_ts}: {'PASS to L3 (descriptive only)' if verdict['passed'] else 'FAIL stays L2'}."
            if tag not in r["notes"]:
                r["notes"] = f"{r['notes']} {tag}".strip()
        out.append(r)
    return out

acq = read_csv_rows(ACQUISITION_PATH)
acq = apply_audit(acq, "osm_network", osm_verdict)
acq = apply_audit(acq, "os_open_greenspace", gs_verdict)
write_csv_rows(ACQUISITION_PATH, acq, list(acq[0].keys()))

# Upsert a Gate 0B row into the evidence-gate ledger.
GATE_PATH = DATA / "evidence_gate_status_phase1.csv"
gate_rows = read_csv_rows(GATE_PATH)
g0b = {
    "gate_id": "G0B",
    "gate_name": "Raw acquisition and quality audit",
    "status": "in_progress",
    "owner": "SpineLens",
    "started_on": "2026-06-03",
    "completed_on": "",
    "exit_criteria": "Priority raw extracts acquired with checksums; CRS, validity, connectivity, coverage audited; cross-checks logged; audit note written",
    "next_action": "Cross-check OSM against OS authoritative roads/basemap and refine provisional anchors before route legibility modelling",
    "notes": "OSM walk graph and OS Open Greenspace audited to Level 3 (descriptive only); no funding claim",
}
if any(r["gate_id"] == "G0B" for r in gate_rows):
    gate_rows = [g0b if r["gate_id"] == "G0B" else r for r in gate_rows]
else:
    gate_rows.append(g0b)
write_csv_rows(GATE_PATH, gate_rows, list(gate_rows[0].keys()))

display(pd.DataFrame(acq).query("source_id in ['osm_network', 'os_open_greenspace']")[
    ["source_id", "forensic_status", "evidence_level", "quality_audited", "can_support_funding_claim"]])

In [ ]:
# Funding-safe artifact: the Gate 0B Quality Audit Note.
lines = [
    "# Gate 0B Quality Audit Note",
    "",
    f"Generated: {audit_ts}",
    "",
    "## Decision",
    "",
    f"OSM walk network data quality: {'PASS -> Evidence Level 3 (descriptive claims only).' if osm_verdict['passed'] else 'FAIL -> stays Evidence Level 2: ' + ', '.join(osm_verdict['failed_checks']) + '.'}",
    f"OS Open Greenspace data quality: {'PASS -> Evidence Level 3 (descriptive claims only).' if gs_verdict['passed'] else 'FAIL -> stays Evidence Level 2: ' + ', '.join(gs_verdict['failed_checks']) + '.'}",
    "",
    "Data-quality verdicts judge the datasets themselves. Site/context observations",
    "(e.g. whether greenspace exists at the pavilion) are reported as findings below and",
    "do not, by themselves, fail a dataset. No source can support funding-facing claims",
    "yet (that needs Levels 4-5: cross-check and field validation).",
    "",
    "## Data Integrity",
    "",
    "| Check | OSM walk network | OS Open Greenspace |",
    "|---|---|---|",
    f"| CRS | {osm_crs} | {gs_crs} |",
    f"| Invalid geometries | {edge_invalid} | {gs_invalid} |",
    f"| Empty geometries | {edge_empty} | {gs_empty} |",
    f"| Features | {len(audit_edges)} edges / {len(audit_nodes)} nodes | {len(greenspace)} sites |",
    "",
    "## Network Connectivity",
    "",
    f"- Connected components: {connectivity['components']}",
    f"- Largest component: {connectivity['largest_component_nodes']} nodes ({connectivity['largest_component_fraction']:.1%})",
    f"- Nodes outside largest component: {connectivity['nodes_outside_largest']}",
    f"- Dead-ends (degree <= 1): {connectivity['dead_ends']}",
    f"- Decision nodes (degree >= 4): {connectivity['decision_nodes_deg_ge_4']}",
    f"- Total walk-network length: {round(edge_len['total'] / 1000, 2)} km",
    f"- Node coverage inside Gate 0 boundary: {coverage['coverage_fraction']:.1%}",
    f"- Key origins + gateway in one connected component (routing feasible): {key_anchors_connected}",
    f"- Key anchors stranded in disconnected fragments: {', '.join(disconnected_key_anchors) if disconnected_key_anchors else 'none'}",
    "",
    "## Anchor Snapping",
    "",
    "| Anchor | Snap distance (m) | Quality |",
    "|---|---:|---|",
]
for _, row in snap_df.sort_values("snap_distance_m").iterrows():
    lines.append(f"| {row['anchor_id']} | {row['snap_distance_m']} | {row['snap_quality']} |")
lines += [
    "",
    f"Poor snaps needing anchor refinement: {', '.join(poor) if poor else 'none'}.",
    "",
    "## Cross-Source Findings",
    "",
    f"- Route-family references that do NOT resolve to a known anchor ID: {', '.join(unresolved_families) if unresolved_families else 'none'}.",
    f"- Greenspace sites within the study boundary: {gs_in_boundary}.",
    f"- Greenspace sites within 250 m of the Ryder Street pavilion search area: {len(gs_near)}.",
    f"- Nearest OS Open Greenspace site to the pavilion search area: {nearest_gs_m} m.",
    "",
    "KEY FINDING: the Ryder Street pavilion search area is not on OS Open Greenspace land",
    "within 250 m. The deck's 'grassland' read must be validated against aerial imagery and",
    "ownership data; it is not corroborated by this authoritative open-space dataset.",
    "",
    f"KEY FINDING: route-modelling is blocked for {', '.join(disconnected_key_anchors) if disconnected_key_anchors else 'no anchors'} -",
    "the snapped node sits in a disconnected network fragment, so no route can yet be computed",
    "to the gateway. Refine the anchor or repair/extend the network before route modelling.",
    "",
    "KEY FINDING: two experimental route families (New Street, Snow Hill) reference origin IDs",
    "that do not exist in the anchor table. The route ledger and anchor table must be reconciled.",
    "",
    "## Visual Evidence",
    "",
    "- outputs/reports/gate0b_audit_media/fig1_study_area_overview.png",
    "- outputs/reports/gate0b_audit_media/fig2_connectivity.png",
    "- outputs/reports/gate0b_audit_media/fig3_distributions.png",
    "- outputs/reports/gate0b_audit_media/fig4_gateway_barrier_context.png",
    "- outputs/maps/gate0b_quality_audit.html",
    "",
    "## Caveats",
    "",
    "- OSM is volunteered data; it remains a proxy until cross-checked against OS authoritative sources.",
    "- All anchors are provisional and not field-validated.",
    "- Greenspace presence is a context layer, not a land-ownership or buildability conclusion.",
    "",
    "## Next Controlled Step",
    "",
    "Resolve the route-family / anchor ID mismatch, refine the Snow Hill anchor onto the connected",
    "network, validate the pavilion search area against aerial imagery and ownership data, and",
    "cross-check the OSM network against OS Open Roads / OS OpenMap Local before route-legibility modelling.",
]
note = "\n".join(lines)
(REPORTS / "gate0b_quality_audit_note.md").write_text(note, encoding="utf-8")
print(note)

## What this unlocks

Audited sources can now support **descriptive** statements (Level 3), for example:
"the acquired walk network covers the study corridor and is well connected." They
**cannot** yet support diagnostic, feasibility, recommendation, or impact claims.

The next notebook step is route-legibility modelling, but only after the cross-source
checks above are actioned (resolve the route-family ID mismatch and refine provisional
anchors).